# 01. Preliminaries (Sec. II)

This notebook runs each building block of QPASE once and prints what it produces: the ring $R_q$,
the gadget $G$, the Banerjee-Peikert lattice PRF, Ding's robust extractor, the DKG, the variant TOPRF
$\Pi_{TOPRF}$ and the lattice HKDF. Every value comes from `ris_sem_project`. The notebook only calls
those functions and prints their results.

In [1]:
import itertools
import statistics
import time

import numpy as np

from ris_sem_project.display import bits, peek, table
from ris_sem_project.params import PARAM_I, PARAM_II, Lattice, setup
from ris_sem_project.primitives import extractor, toprf
from ris_sem_project.primitives.bp_prf import encode_input
from ris_sem_project.primitives.gadget import g_inverse
from ris_sem_project.primitives.hkdf import Hkdf
from ris_sem_project.primitives.prf import prf
from ris_sem_project.primitives.ring import Ring
from ris_sem_project.primitives.sampling import gaussian, uniform
from ris_sem_project.primitives.secret_sharing import (
    NonIntegerLagrangeError,
    genkey,
    genshare,
    lagrange_at_zero,
    reconstruct,
)

## Parameters (Table II and plan decision 7)

Table II fixes $q$ and $n$. The paper leaves $p$, $\sigma$ and $L$ open. The implementation sets
$p = 2$ (the extractor outputs bits), $\sigma = 3.2$ (read as the standard deviation, as the text of
Definition 3 says) and $L = \kappa = 128$. $\ell = \lceil \log_2 q \rceil$. Every demo runs on PARAM I.
PARAM II's ring has odd degree, so $X = -1$ is a root of $X^n + 1$ (see notebook 06).

In [2]:
rows = [
    (p.name, f"q = {p.q}", f"n = {p.n}", f"ell = {Ring(p.q, p.n).ell}", f"p = {p.p}", f"sigma = {p.sigma}", f"L = {p.L}")
    for p in (PARAM_I, PARAM_II)
]
print(table(("set", "q", "n", "ell", "p", "sigma", "L"), rows))
print(f"q == 2^28 - 57: {PARAM_I.q == 2**28 - 57}")

pp = setup(PARAM_I, N=3, t=2, mu=5)
lattice = Lattice.from_pp(pp)
ring = lattice.ring
print(f"pp = Setup(PARAM I): a, a_0, a_1 in R_q^(1 x ell), each {peek(pp.a)}")

set       q              n        ell       p      sigma        L
--------  -------------  -------  --------  -----  -----------  -------
PARAM I   q = 268435399  n = 512  ell = 28  p = 2  sigma = 3.2  L = 128
PARAM II  q = 268435399  n = 595  ell = 28  p = 2  sigma = 3.2  L = 128
q == 2^28 - 57: True
pp = Setup(PARAM I): a, a_0, a_1 in R_q^(1 x ell), each 7fc4…650f (57344 B)


## The ring $R_q = \mathbb{Z}_q[X]/(X^n + 1)$ (Sec. II-B)

$q = 2^{28} - 57$ has no $2n$-th roots of unity, so `Ring.mul` multiplies through a complex FFT on
14-bit limbs (plan decision 2). The cell checks that $X^{n-1} \cdot X = -1$ and compares three random
products with a schoolbook product computed here in exact Python integers, used only as a reference
for this check.

In [3]:
X = np.zeros(ring.n, dtype=np.int64)
X[1] = 1
X_top = np.zeros(ring.n, dtype=np.int64)
X_top[-1] = 1
wrapped = ring.centered(ring.mul(X_top, X))
print(f"X^(n-1) * X has constant coefficient {wrapped[0]} and {np.count_nonzero(wrapped[1:])} other nonzero coefficients")

exact_products = 0
for _ in range(3):
    a, b = uniform(ring, (ring.n,)), uniform(ring, (ring.n,))
    full = np.convolve(a.astype(object), b.astype(object))
    reference = (full[: ring.n] - np.append(full[ring.n :], 0)) % ring.q
    exact_products += ring.mul(a, b).tolist() == reference.tolist()
print(f"{exact_products}/3 random products in R_q equal the exact integer reference")

X^(n-1) * X has constant coefficient -1 and 0 other nonzero coefficients
3/3 random products in R_q equal the exact integer reference


## The gadget $G$ and $G^{-1}$ (Sec. II-B)

$G = (1, 2, \ldots, 2^{\ell-1})$. For $v \in R_q^{1 \times \ell}$, $G^{-1}(v) \in R_q^{\ell \times \ell}$
is the binary matrix $B$ with $G B = v$: row $k$ of $B$ holds bit $k$ of every coefficient.

In [4]:
v = uniform(ring, (ring.ell, ring.n))
B = g_inverse(ring, v)
G = 2 ** np.arange(ring.ell)
recomposed = np.einsum("k,kjn->jn", G, B.astype(np.int64))
print(f"v: shape {v.shape}, G^-1(v): shape {B.shape}, entries {sorted(set(np.unique(B).tolist()))}")
print(f"G G^-1(v) == v: {np.array_equal(recomposed, v)}")
gadget_identity = np.array_equal(recomposed, v)

v: shape (28, 512), G^-1(v): shape (28, 28, 512), entries [0, 1]
G G^-1(v) == v: True


## The BP lattice PRF (Lemma 1, [80])

For $x \in \{0,1\}^L$, $a_x = a_{x_1} G^{-1}(a_{x_2} G^{-1}(\cdots a_{x_L}))$ and
$F_k(x) = \lfloor (p/q) \cdot a_x k \rceil$. With $p = 2$, each of the $\ell n$ coefficients of
$a_x k$ rounds to one bit. The input $x$ is the first $L$ bits of SHA3-256 of the data.

In [5]:
x = encode_input(b"correct horse battery staple", pp.L)
x_other = encode_input(b"correct horse battery stapler", pp.L)
k = uniform(ring, (ring.n,))
a_x = lattice.prf.a_x(x)
F1 = lattice.prf.evaluate(k, x)
F2 = lattice.prf.evaluate(k, x)
F_other = lattice.prf.evaluate(k, x_other)
print(f"x: {''.join(map(str, x[:32]))}... ({x.size} bits)")
print(f"a_x: shape {a_x.shape}, {peek(ring.to_bytes(a_x))}")
print(f"F_k(x): {bits(ring.pack_bits(F1))}")
print(f"F_k(x) has {F1.size} bits = ell * n = {ring.ell} * {ring.n}")
print(f"F_k(x) twice, same k and x: identical = {np.array_equal(F1, F2)}")
print(f"F_k(x') for a one-character change: {np.count_nonzero(F1 != F_other)} of {F1.size} bits differ")

x: 10101111100110101100001111011010... (128 bits)
a_x: shape (28, 512), 42f2…f807 (57344 B)
F_k(x): 14336 bits, 7249 ones: 1110100011100000…0001110100111110
F_k(x) has 14336 bits = ell * n = 28 * 512
F_k(x) twice, same k and x: identical = True
F_k(x') for a one-character change: 7129 of 14336 bits differ


Each evaluation runs $L - 1 = 127$ gadget steps, each a $1 \times \ell$ by $\ell \times \ell$ product
in $R_q$, then one product with $k$. The cell times `BPPRF.evaluate` with `time.perf_counter`.

In [6]:
runs = 7
times = []
for _ in range(runs):
    start = time.perf_counter()
    lattice.prf.evaluate(k, x)
    times.append(time.perf_counter() - start)
print(f"BP PRF F_k(x) over {runs} runs: median {statistics.median(times):.2f} s, min {min(times):.2f} s, max {max(times):.2f} s")

BP PRF F_k(x) over 7 runs: median 0.66 s, min 0.65 s, max 0.66 s


## Ding's robust extractor (Definition 8, [69])

$\sigma \leftarrow S(y)$ is one hint bit per coefficient. $E(x, \sigma) = (x + \sigma (q-1)/2 \bmod q) \bmod 2$
on centered representatives. If $x - y$ is even and small, $E(x, \sigma) = E(y, \sigma)$. The cell
perturbs $y$ by $2e$ with $e \leftarrow \chi_\sigma$, then by $2e + 1$ for contrast.

In [7]:
y = uniform(ring, (ring.ell, ring.n))
hint = extractor.hint(ring, y)
e = gaussian(ring, y.shape, pp.sigma)
x_even = ring.add(y, 2 * e)
x_odd = ring.add(x_even, np.ones_like(y))
E_y = extractor.extract(ring, y, hint)
E_even = extractor.extract(ring, x_even, hint)
E_odd = extractor.extract(ring, x_odd, hint)
print(f"hint sigma = S(y): {bits(ring.pack_bits(hint))}")
print(f"perturbation 2e: max |2e| = {np.abs(ring.centered(2 * e)).max()}")
print(f"E(y + 2e, sigma) == E(y, sigma): {np.array_equal(E_even, E_y)}")
print(f"E(y + 2e + 1, sigma): {np.count_nonzero(E_odd != E_y)} of {E_y.size} bits differ from E(y, sigma)")

hint sigma = S(y): 14336 bits, 7087 ones: 0100001100010110…0111011111010000
perturbation 2e: max |2e| = 24
E(y + 2e, sigma) == E(y, sigma): True
E(y + 2e + 1, sigma): 14336 of 14336 bits differ from E(y, sigma)


## Distributed key generation, $N = 3$, $t = 2$ (Definitions 11-12, [83])

Genshare at $S_i$: $f_i(X) = s_i + c_1 X$ with $s_i \leftarrow \chi_\sigma$ and $c_1$ uniform in $R_q$
(degree $t - 1$, plan decision 5). $S_i$ sends $[s_i]_j = f_i(j)$ to each $S_j$. Genkey at $S_j$:
$k_j = \sum_i [s_i]_j$. The master key $K = \sum_i s_i$ is never formed by any server.

In [8]:
ids = (1, 2, 3)
dealt = {i: genshare(ring, 2, ids, pp.sigma) for i in ids}
keys = {j: genkey(ring, [dealt[i][j] for i in ids]) for j in ids}
print(table(("dealer S_i", "[s_i]_1", "[s_i]_2", "[s_i]_3"), [(f"S_{i}", *(peek(ring.to_bytes(dealt[i][j])) for j in ids)) for i in ids]))
print()
print(table(("server", "k_j = sum_i [s_i]_j"), [(f"S_{j}", peek(ring.to_bytes(keys[j]))) for j in ids]))

dealer S_i  [s_i]_1             [s_i]_2             [s_i]_3
----------  ------------------  ------------------  ------------------
S_1         b11a…e700 (2048 B)  6035…cf01 (2048 B)  0f50…b702 (2048 B)
S_2         4169…5c06 (2048 B)  b9d2…b90c (2048 B)  313c…1603 (2048 B)
S_3         955c…cd0a (2048 B)  24b9…9a05 (2048 B)  b315…6700 (2048 B)

server  k_j = sum_i [s_i]_j
------  -------------------
S_1     c0e0…1102 (2048 B)
S_2     afc1…2304 (2048 B)
S_3     65a2…3406 (2048 B)


The user combines TOPRF answers with the Lagrange coefficients at 0,
$\lambda_i = \prod_{j \ne i} j / (j - i)$. They must be integers so that the noise term
$\sum_i \lambda_i e'_i$ stays small (plan decision 3). The cell lists every subset of at least $t$
servers. For this demo only, it then reconstructs $K$ from the shares to show that $K$ is small.

In [9]:
lagrange_rows = []
integer_subsets = []
for size in (2, 3):
    for subset in itertools.combinations(ids, size):
        try:
            lam = lagrange_at_zero(subset)
            integer_subsets.append(subset)
            lagrange_rows.append((subset, lam, "yes"))
        except NonIntegerLagrangeError as error:
            lagrange_rows.append((subset, str(error).split(" for ")[0], "no"))
print(table(("subset", "lambda_i", "integer"), lagrange_rows))

K_from = {subset: reconstruct(ring, {j: keys[j] for j in subset}) for subset in integer_subsets}
K_sum = sum((reconstruct(ring, {j: dealt[i][j] for j in (1, 2)}) for i in ids), np.zeros(ring.n, dtype=np.int64)) % ring.q
K = K_from[(1, 2)]
print()
print(f"K from {', '.join(map(str, integer_subsets))} identical: {all(np.array_equal(v, K) for v in K_from.values())}")
print(f"K == s_1 + s_2 + s_3: {np.array_equal(K, K_sum)}")
print(f"max |K coefficient| = {np.abs(ring.centered(K)).max()}, max |k_1 coefficient| = {np.abs(ring.centered(keys[1])).max()} (q/2 = {ring.q // 2})")

subset     lambda_i             integer
---------  -------------------  -------
(1, 2)     {1: 2, 2: -1}        yes
(1, 3)     lambda_1 = 3/2       no
(2, 3)     {2: 3, 3: -2}        yes
(1, 2, 3)  {1: 3, 2: -3, 3: 1}  yes

K from (1, 2), (2, 3), (1, 2, 3) identical: True
K == s_1 + s_2 + s_3: True
max |K coefficient| = 16, max |k_1 coefficient| = 134086118 (q/2 = 134217699)


## One run of the variant TOPRF (Fig. 1-2, Theorem 1)

U: $b = a r + a_x + 2e$ with $r, e \leftarrow \chi_\sigma$. $S_i$: $b_{k_i} = b k_i + 2e'_i$ and
$pk_i = a k_i + 2e_i$. U: $F = \sum_i \lambda_i b_{k_i} - r \sum_i \lambda_i pk_i = a_x K + 2e''$ with
$e'' = eK + \sum_i \lambda_i e'_i - r \sum_i \lambda_i e_i$. U extracts $K_u = E(F, \sigma)$ with
$\sigma = S(F)$ at Register, and reuses that $\sigma$ at every later run (plan decision 4).
The cell uses servers $\{1, 2\}$ and computes $a_x K$ directly, with the $K$ from the cell above, to
print the noise.

In [10]:
password_x = encode_input(b"correct horse battery staple", pp.L)
ax = lattice.prf.a_x(password_x)
server_pks = {i: toprf.server_public_key(ring, lattice.a, keys[i], pp.sigma) for i in ids}


def toprf_run(subset):
    b, r = toprf.blind(ring, lattice.a, ax, pp.sigma)
    evaluations = {i: toprf.evaluate(ring, b, keys[i], pp.sigma) for i in subset}
    return b, toprf.combine(ring, evaluations, {i: server_pks[i] for i in subset}, r)


b, F_reg = toprf_run((1, 2))
noise = ring.centered(ring.sub(F_reg, ring.mul(ax, K)))
k_u, sigma_hint = toprf.finalize(ring, F_reg)
print(f"U -> S_1, S_2: b = {peek(ring.to_bytes(b))}")
print(f"F - a_x K = 2e'': all even = {bool((noise % 2 == 0).all())}, max |2e''| = {np.abs(noise).max()}, q/4 = {ring.q // 4}")
print(f"K_u = E(F, sigma): {bits(k_u)}")

_, F_login = toprf_run((2, 3))
k_u_again, _ = toprf.finalize(ring, F_login, sigma_hint)
drift = ring.centered(ring.sub(F_login, F_reg))
print(f"second run with fresh r, e on {{2, 3}}: F' - F all even = {bool((drift % 2 == 0).all())}, max |F' - F| = {np.abs(drift).max()}")
print(f"K_u' = E(F', sigma) == K_u: {k_u_again == k_u}")

U -> S_1, S_2: b = 4fcd…ba04 (57344 B)
F - a_x K = 2e'': all even = True, max |2e''| = 6240, q/4 = 67108849
K_u = E(F, sigma): 14336 bits, 7147 ones: 0001000000011011…1011110000101110
second run with fresh r, e on {2, 3}: F' - F all even = True, max |F' - F| = 10210
K_u' = E(F', sigma) == K_u: True


## Lattice HKDF and PRF (Definitions 9-10, plan Part 3)

HKDF is Krawczyk's extract-then-expand with the BP PRF in place of HMAC:
$PRK = F_{K_u}(\text{salt})$, then $dsk = F_{PRK}(w \| 1)$. The PRF of Setup is the same BP PRF,
$v = PRF(dsk, \rho)$. A key is an $\ell n$-bit string that packs into one element of $R_q$.

In [11]:
hkdf = Hkdf(lattice.prf, k_u)
dsk = hkdf.derive(b"QPASE/dsk/medical")
v_out = prf(lattice.prf, dsk, b"QPASE/v/" + (12345).to_bytes(4, "big"))
print(f"K_u:                 {peek(k_u)}")
print(f"dsk = HKDF(K_u, w):  {peek(dsk)}, {8 * len(dsk)} bits")
print(f"v = PRF(dsk, rho):   {peek(v_out)}")
print(f"HKDF again gives the same dsk: {Hkdf(lattice.prf, k_u).derive(b'QPASE/dsk/medical') == dsk}")

K_u:                 101b…bc2e (1792 B)
dsk = HKDF(K_u, w):  867e…e4db (1792 B), 14336 bits
v = PRF(dsk, rho):   d568…9690 (1792 B)


HKDF again gives the same dsk: True


## Checks

These assertions fail the notebook run if any property shown above did not hold.

In [12]:
assert wrapped[0] == -1 and not wrapped[1:].any(), "X^n = -1 in R_q"
assert exact_products == 3, "R_q products are exact"
assert gadget_identity, "G G^-1(v) = v"
assert np.array_equal(F1, F2) and F1.size == ring.ell * ring.n, "BP PRF is deterministic with ell * n output bits"
assert np.array_equal(E_even, E_y), "an even, small perturbation does not change E(., sigma)"
assert integer_subsets == [(1, 2), (2, 3), (1, 2, 3)], "{1, 3} is the only subset with non-integer Lagrange coefficients"
assert (noise % 2 == 0).all() and np.abs(noise).max() < ring.q // 4, "TOPRF noise 2e'' is even and far below q/4"
assert k_u_again == k_u and len(k_u) == ring.ell * ring.n // 8, "the stored hint reproduces K_u"
assert len(dsk) == len(v_out) == len(k_u), "HKDF and PRF outputs are ell * n bits"
print("all preliminaries checks passed")

all preliminaries checks passed
